# 06 · Camada Gold — Continuidade

## 1. Objetivo e método

A Silver entregou o indicador no grão do conjunto. Esta etapa agrega até a distribuidora, aplica o recorte da análise e produz a evolução que responde à pergunta de negócio.

### A agregação é normativa

O PRODIST Módulo 8 não deixa a agregação a critério de quem calcula. As equações 41 a 46 definem os dois passos:

| Passo | Equações | Fórmula |
|---|---|---|
| Global mensal | 41 a 43 | `DECGn = Σ(DECi × NUCi) / NUCGn`, com `NUCGn = Σ NUCi` |
| Agregação anual | 44 a 46 | `DECGk = Σ(DECGn × NUCGn) / NUCGk`, com `NUCGk = (Σ NUCGn) / k` |

No primeiro passo, cada conjunto entra ponderado pelo número de unidades consumidoras do próprio mês. No segundo, o denominador é a média mensal do universo, não a soma. A simplificação corrente de somar os doze meses só coincide com a norma quando o universo não varia ao longo do ano.

A norma exige duas casas decimais em cada etapa, e o arredondamento é aplicado como ela determina.

### O recorte entra aqui

A Silver guarda a série inteira e o porte como atributo. É nesta etapa que o universo se fecha:

- apenas distribuidoras marcadas como `grande_porte`, critério medido em dezembro de 2022
- apenas anos civis completos, avaliados no nível da distribuidora. O ano incompleto é descartado sozinho, sem eliminar a empresa dos demais
- janela de 2022 a 2025

### Tabelas produzidas

| Tabela | Grão | Papel |
|---|---|---|
| `fato_continuidade_global_mensal` | distribuidora, ano, mês | Indicador da distribuidora mês a mês |
| `fato_continuidade_anual` | distribuidora, ano | Indicador anual pela agregação normativa |
| `ranking_evolucao` | distribuidora | Variação entre o primeiro e o último ano da janela |


## 2. Configuração

In [0]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_SILVER, SCHEMA_GOLD

SILVER = f"{CATALOG}.{SCHEMA_SILVER}"
GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

ANO_INICIAL = 2022
ANO_FINAL = 2025
ANOS_JANELA = list(range(ANO_INICIAL, ANO_FINAL + 1))

# The norm states two decimal places at each aggregation step.
CASAS = 2

# Indicators carried through the whole layer: the normative pair and the own pair.
INDICADORES = ["dec", "fec", "dec_fi", "fec_fi"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_GOLD}")

print(f"Origem..........: {SILVER}")
print(f"Destino.........: {GOLD}")
print(f"Janela..........: {ANO_INICIAL} a {ANO_FINAL}")
print(f"Indicadores.....: {', '.join(INDICADORES)}")

## 3. Universo e recorte

Duas etapas: fechar o universo pelo porte e descartar os anos incompletos.

#### Aplicar o recorte:

O fato da Silver traz a série inteira e todas as distribuidoras. Aqui entram os três filtros da análise — janela, porte e ano completo — e cada um informa quanto removeu, para que o recorte seja audítavel em vez de silencioso.

O descarte do ano incompleto é por distribuidora e por ano. Uma empresa que falhou no envio de um ano permanece nos demais: eliminar a empresa inteira por um envio falho descartaria histórico bom por causa de um evento pontual.

In [0]:
fato = spark.table(f"{SILVER}.fato_continuidade_mensal")
dim_dx = spark.table(f"{SILVER}.dim_distribuidora")

universo = dim_dx.filter(F.col("grande_porte")).select("num_cnpj", "sig_agente")
n_universo = universo.count()

na_janela = fato.filter(F.col("ano").isin(ANOS_JANELA))
do_universo = na_janela.join(universo, "num_cnpj", "inner")
completo = do_universo.filter(F.col("ano_completo"))

# Which distributor-years the completeness filter removes, if any.
descartados = (do_universo
    .filter(~F.col("ano_completo"))
    .groupBy("sig_agente", "num_cnpj", "ano")
    .agg(F.max("meses_no_ano").alias("meses_no_ano"))
    .orderBy("sig_agente", "ano"))

n_descartados = descartados.count()

print(f"linhas na Silver...........: {fato.count():,}")
print(f"dentro da janela...........: {na_janela.count():,}")
print(f"do universo de grande porte: {do_universo.count():,}")
print(f"com ano completo...........: {completo.count():,}")
print(f"distribuidoras no universo.: {n_universo}")
print(f"distribuidora-ano descartado: {n_descartados}")

if n_descartados:
    display(descartados)

## 4. Indicador global mensal

Equações 41 a 43 do PRODIST Módulo 8.

#### Agregar conjuntos em distribuidora:

Cada conjunto entra ponderado pelo seu próprio número de unidades consumidoras no mês, e o denominador é a soma desses números. É a média ponderada que a norma define: um conjunto com trezentos mil consumidores pesa trezentas vezes mais que um com mil.

O peso vem de `num_con`, publicado na mesma base do indicador. Nenhuma contagem de unidades consumidoras de outra fonte entra no cálculo, porque as bases da ANEEL medem universos distintos e a troca introduziria erro maior que a divergência observada entre elas.

In [0]:
# Equations 41 to 43: weighted mean of the sets, weighted by that month's NUC.
ponderados = completo
for nome in INDICADORES:
    ponderados = ponderados.withColumn(f"_p_{nome}", F.col(nome) * F.col("num_con"))

agregados = (ponderados
    .groupBy("num_cnpj", "ano", "mes")
    .agg(F.sum("num_con").alias("nuc_g"),
         F.countDistinct("ide_conjunto").alias("conjuntos"),
         *[F.sum(f"_p_{nome}").alias(f"_s_{nome}") for nome in INDICADORES]))

global_mensal = agregados
for nome in INDICADORES:
    global_mensal = global_mensal.withColumn(
        nome, F.round(F.col(f"_s_{nome}") / F.col("nuc_g"), CASAS))

global_mensal = (global_mensal
    .join(universo, "num_cnpj")
    .select("num_cnpj", "sig_agente", "ano", "mes", "conjuntos",
            F.col("nuc_g").cast("long").alias("nuc_g"), *INDICADORES))

(global_mensal.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fato_continuidade_global_mensal"))

print(f"fato_continuidade_global_mensal: {global_mensal.count():,} linhas")
display(global_mensal.orderBy("sig_agente", "ano", "mes").limit(15))

## 5. Agregação anual

Equações 44 a 46 do PRODIST Módulo 8.

#### Agregar meses em ano:

O numerador soma o produto do indicador mensal pelo universo daquele mês; o denominador é a média mensal do universo no ano, e não a soma. É essa média que faz a expressão degenerar na soma simples dos doze meses quando o número de consumidores não varia — e que a separa dela quando varia.

A diferença entre os dois caminhos é calculada ao lado, para dimensionar quanto a simplificação corrente de mercado distorceria o resultado nesta base.

In [0]:
# Equations 44 to 46: the denominator is the monthly average of the universe.
ponderados_ano = global_mensal
for nome in INDICADORES:
    ponderados_ano = ponderados_ano.withColumn(
        f"_p_{nome}", F.col(nome) * F.col("nuc_g"))

anual = (ponderados_ano
    .groupBy("num_cnpj", "sig_agente", "ano")
    .agg(F.count("*").alias("meses"),
         F.sum("nuc_g").alias("_soma_nuc"),
         *[F.sum(f"_p_{nome}").alias(f"_s_{nome}") for nome in INDICADORES],
         *[F.sum(nome).alias(f"_simples_{nome}") for nome in INDICADORES]))

anual = anual.withColumn("nuc_gk", F.round(F.col("_soma_nuc") / F.col("meses"), 0))

for nome in INDICADORES:
    anual = anual.withColumn(nome, F.round(F.col(f"_s_{nome}") / F.col("nuc_gk"), CASAS))

# How far the common shortcut of simply adding the months would land.
anual = (anual
    .withColumn("dec_fi_soma_simples", F.round(F.col("_simples_dec_fi"), CASAS))
    .withColumn("dif_vs_soma_simples",
                F.round(F.col("dec_fi") - F.col("_simples_dec_fi"), CASAS)))

anual = anual.select("num_cnpj", "sig_agente", "ano", "meses",
                     F.col("nuc_gk").cast("long").alias("nuc_gk"),
                     *INDICADORES, "dec_fi_soma_simples", "dif_vs_soma_simples")

(anual.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fato_continuidade_anual"))

print(f"fato_continuidade_anual: {anual.count():,} linhas")
display(anual.orderBy(F.abs(F.col("dif_vs_soma_simples")).desc()).limit(15))

#### Leitura da diferença entre os dois caminhos:

A coluna `dif_vs_soma_simples` mede, em horas, quanto o indicador anual se afasta da soma direta dos doze meses. Valor próximo de zero significa universo estável no ano; valor relevante indica distribuidora cujo número de consumidores variou o bastante para que a ponderação mensal importe.

A tabela está ordenada pela diferença absoluta, então o topo mostra onde a simplificação corrente falharia mais.

## 6. Evolução e ranking

A resposta à pergunta de negócio.

#### Calcular a evolução:

A comparação é entre o primeiro e o último ano da janela. Redução de DEC-FI significa melhora: menos horas sem energia por consumidor.

Duas medidas, porque uma sozinha engana:

- a variação absoluta, em horas, diz o que o consumidor sentiu
- a variação percentual diz o esforço relativo, e favorece quem partia de patamar alto

Uma distribuidora que caiu de 20 para 15 horas melhorou 5 horas e 25%; outra que caiu de 4 para 3 melhorou 1 hora e 25%. O ranking usa o percentual, com a variação absoluta ao lado, e a distinção fica registrada.

Distribuidora que não tenha os dois anos extremos — por ano incompleto descartado no recorte — fica fora do ranking, e a ausência é informada.

In [0]:
inicio = (anual.filter(F.col("ano") == ANO_INICIAL)
    .select("num_cnpj", "sig_agente",
            *[F.col(n).alias(f"{n}_inicio") for n in INDICADORES],
            F.col("nuc_gk").alias("nuc_inicio")))

fim = (anual.filter(F.col("ano") == ANO_FINAL)
    .select("num_cnpj",
            *[F.col(n).alias(f"{n}_fim") for n in INDICADORES],
            F.col("nuc_gk").alias("nuc_fim")))

ranking = inicio.join(fim, "num_cnpj", "inner")

for nome in INDICADORES:
    ranking = (ranking
        .withColumn(f"{nome}_var_abs",
                    F.round(F.col(f"{nome}_fim") - F.col(f"{nome}_inicio"), CASAS))
        # A zero at the start would make the percentage meaningless, not infinite.
        .withColumn(f"{nome}_var_pct",
                    F.when(F.col(f"{nome}_inicio") > 0,
                           F.round((F.col(f"{nome}_fim") / F.col(f"{nome}_inicio") - 1) * 100,
                                   CASAS))))

janela_ranking = Window.orderBy(F.col("dec_fi_var_pct").asc())
ranking = (ranking
    .withColumn("posicao", F.row_number().over(janela_ranking))
    .withColumn("melhorou", F.col("dec_fi_var_abs") < 0))

(ranking.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.ranking_evolucao"))

no_ranking = ranking.count()
fora = universo.count() - no_ranking

print(f"distribuidoras no ranking..: {no_ranking}")
print(f"fora por ano faltante......: {fora}")
print(f"melhoraram o DEC-FI........: {ranking.filter('melhorou').count()}")

display(ranking
    .select("posicao", "sig_agente", "dec_fi_inicio", "dec_fi_fim",
            "dec_fi_var_abs", "dec_fi_var_pct",
            "fec_fi_inicio", "fec_fi_fim", "fec_fi_var_pct",
            F.col("nuc_fim").alias("ucs"))
    .orderBy("posicao"))

#### Visualizar o ranking:

O gráfico ordena as distribuidoras pela variação percentual do DEC-FI. Barra à esquerda do zero indica redução da duração de interrupção, ou seja, melhora percebida pelo consumidor.

In [0]:
# Horizontal bars with seaborn: the ranking has a few dozen rows, so the whole
# result fits in the driver and sns.barplot can take the dataframe directly.
import matplotlib.pyplot as plt
import seaborn as sns

dados = (ranking
    .select("sig_agente", "dec_fi_var_pct", "melhorou")
    .orderBy(F.col("dec_fi_var_pct").asc())
    .toPandas())

sns.set_theme(style="whitegrid", context="notebook")

fig, ax = plt.subplots(figsize=(8, max(4, len(dados) * 0.30)))
sns.barplot(data=dados, x="dec_fi_var_pct", y="sig_agente", hue="melhorou",
            palette={True: "#2e7d32", False: "#c62828"}, dodge=False,
            legend=False, ax=ax)

ax.axvline(0, color="#37474f", linewidth=1)
ax.set_xlabel(f"Variacao do DEC-FI entre {ANO_INICIAL} e {ANO_FINAL} (%)")
ax.set_ylabel("")
ax.set_title("Evolucao da continuidade - distribuidoras de grande porte")
ax.tick_params(axis="y", labelsize=8)
sns.despine(fig=fig, left=True)
fig.tight_layout()
display(fig)
plt.close(fig)

In [0]:
COMENTARIOS = {
    f"{GOLD}.fato_continuidade_global_mensal": (
        "Indicador de continuidade da distribuidora no grao mensal, agregado dos conjuntos "
        "pelas Equacoes 41 a 43 do PRODIST Modulo 8. Restrito as distribuidoras de grande "
        "porte e aos anos civis completos.",
        {
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "sig_agente": "Sigla da distribuidora conforme publicada pela ANEEL",
            "ano": "Ano civil de apuracao",
            "mes": "Mes civil de apuracao, de 1 a 12",
            "conjuntos": "Quantidade de conjuntos agregados no mes",
            "nuc_g": "Universo do mes: soma das unidades consumidoras dos conjuntos (Eq. 43)",
            "dec": "DEC normativo da distribuidora no mes, duas casas decimais",
            "fec": "FEC normativo da distribuidora no mes, duas casas decimais",
            "dec_fi": "DEC de falha interna da distribuidora no mes, indicador proprio",
            "fec_fi": "FEC de falha interna da distribuidora no mes, indicador proprio",
        },
    ),
    f"{GOLD}.fato_continuidade_anual": (
        "Indicador anual da distribuidora pelas Equacoes 44 a 46 do PRODIST Modulo 8, com "
        "denominador igual a media mensal do universo. Traz ao lado a soma simples dos meses, "
        "para dimensionar a distorcao da simplificacao corrente de mercado.",
        {
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "sig_agente": "Sigla da distribuidora conforme publicada pela ANEEL",
            "ano": "Ano civil de apuracao",
            "meses": "Meses que compoem o ano; doze quando o ano esta completo",
            "nuc_gk": "Media mensal das unidades consumidoras no ano (Eq. 46)",
            "dec": "DEC normativo anual, duas casas decimais",
            "fec": "FEC normativo anual, duas casas decimais",
            "dec_fi": "DEC de falha interna anual, indicador proprio do trabalho",
            "fec_fi": "FEC de falha interna anual, indicador proprio do trabalho",
            "dec_fi_soma_simples": "DEC-FI pela soma direta dos meses, sem ponderacao",
            "dif_vs_soma_simples": "Diferenca entre o calculo normativo e a soma simples",
        },
    ),
    f"{GOLD}.ranking_evolucao": (
        "Evolucao da continuidade entre o primeiro e o ultimo ano da janela, por "
        "distribuidora de grande porte. Reducao do DEC-FI indica melhora.",
        {
            "posicao": "Posicao no ranking, ordenada pela variacao percentual do DEC-FI",
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "sig_agente": "Sigla da distribuidora conforme publicada pela ANEEL",
            "melhorou": "Verdadeiro quando o DEC-FI do ultimo ano e menor que o do primeiro",
        },
    ),
}

for tabela, (comentario, colunas) in COMENTARIOS.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    for coluna, texto in colunas.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{texto}'")
    print(f"{tabela}: {len(colunas)} colunas comentadas")

## 7. Validação da Gold

Quatro verificações. As duas primeiras conferem a aritmética contra a norma; as duas últimas, a integridade do recorte.

In [0]:
mensal_lido = spark.table(f"{GOLD}.fato_continuidade_global_mensal")
anual_lido = spark.table(f"{GOLD}.fato_continuidade_anual")
ranking_lido = spark.table(f"{GOLD}.ranking_evolucao")

testes = []

# 1. The monthly indicator has to sit between the smallest and the largest set of the month.
limites = (completo
    .groupBy("num_cnpj", "ano", "mes")
    .agg(F.min("dec_fi").alias("min_conj"), F.max("dec_fi").alias("max_conj")))

fora_limites = (mensal_lido.join(limites, ["num_cnpj", "ano", "mes"])
    .filter((F.col("dec_fi") < F.col("min_conj") - 0.01) |
            (F.col("dec_fi") > F.col("max_conj") + 0.01))
    .count())
testes.append(("media ponderada dentro dos extremos dos conjuntos",
               fora_limites == 0,
               f"{fora_limites:,} meses com indicador fora do intervalo dos conjuntos"))

# 2. Recompute one year by hand and compare with the stored value.
amostra = anual_lido.orderBy(F.col("nuc_gk").desc()).limit(1).collect()[0]
recalculo = (mensal_lido
    .filter((F.col("num_cnpj") == amostra["num_cnpj"]) & (F.col("ano") == amostra["ano"]))
    .agg((F.sum(F.col("dec_fi") * F.col("nuc_g")) /
          (F.sum("nuc_g") / F.count("*"))).alias("v"))
    .collect()[0]["v"])
diferenca = abs(round(recalculo, CASAS) - amostra["dec_fi"])
testes.append((f"recalculo manual de {amostra['sig_agente']} em {amostra['ano']}",
               diferenca <= 0.02,
               f"armazenado {amostra['dec_fi']}, recalculado {round(recalculo, CASAS)}"))

# 3. Every year kept must carry twelve months.
meses_errados = anual_lido.filter(F.col("meses") != 12).count()
testes.append(("todo ano mantido tem doze meses",
               meses_errados == 0,
               f"{meses_errados:,} anos com menos de doze meses"))

# 4. The ranking cannot invent or lose distributors.
dx_anual = anual_lido.select("num_cnpj").distinct().count()
dx_ranking = ranking_lido.count()
testes.append(("ranking contido no universo anual",
               dx_ranking <= dx_anual,
               f"{dx_ranking} no ranking contra {dx_anual} com ano apurado"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<50} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nGold de continuidade validada.")
else:
    print("\nHa teste sem passar; investigar antes de citar o resultado.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Critério de ordenação do ranking | Variação percentual do DEC-FI | Decidir se o trabalho apresenta o ranking pelo percentual, pelo absoluto ou por índice que combine os dois |
| Demais métricas | Apenas continuidade está na Gold | As outras cinco métricas seguem o mesmo desenho; o índice consolidado depende de todas |
| Validação externa | Não feita | Confrontar o ranking com o DGC publicado pela ANEEL, item 818 do Módulo 8 |


## Autoavaliação desta etapa

A preencher após a execução.